# Music, Brain & Wellbeing: Decision Tree Hyperparameter Control and Cross-Validation

This notebook explores techniques to prevent overfitting in Decision Trees using hyperparameter control and validates our models using k-fold cross-validation.

We study:
1. First-principles explanations of model complexity parameters (`max_depth`, `min_samples_split`, `min_samples_leaf`).
2. Constructing controlled Decision Tree experiments.
3. Introducing k-fold cross-validation on the training set to prevent validation bias.
4. Comparing the tuned Decision Tree models against our baselines.



## 1. First-Principles of Model Complexity

### Why did the previous Decision Tree overfit?
A default Decision Tree has no depth limit. During training, the tree splits nodes recursively until every leaf is pure (contains only one observation) or contains fewer than `min_samples_split=2` observations. In a dataset with low signal-to-noise ratio, this unconstrained growth leads to the tree memorizing the specific noise of training samples, creating complex decision boundaries that do not generalize.

### Hyperparameters to Control Complexity:
To prevent overfitting, we restrict the tree's size and split conditions using:
* **`max_depth`**: Limits the maximum number of decision levels (splits) from root to leaf. A shallower tree is less complex because it makes coarser splits.
* **`min_samples_split`**: The minimum number of samples required in a node to split it further. Higher values prevent the model from creating rules that apply to very small numbers of observations.
* **`min_samples_leaf`**: The minimum number of samples required to be at a leaf node. Any split that leaves fewer than this number of samples in either child node will be discarded. This ensures leaves represent a representative sample size.



## 2. Load Data and Split

We load `data/processed/mxmh_cleaned.csv`, split into features (X) and target (y), and apply our 80/20 train/test split.



In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add parent directory to sys.path to allow importing src
sys.path.append(os.path.abspath(".."))

from src.features.preprocessing import build_preprocessor

# Load preprocessed dataset
df = pd.read_csv("../data/processed/mxmh_cleaned.csv")
y = df["Anxiety"]
X = df.drop(columns=["Anxiety", "Depression", "Insomnia", "OCD"])

categorical_cols = list(X.select_dtypes(include=["object", "category"]).columns)
numerical_cols = list(X.select_dtypes(include=["int64", "float64"]).columns)



C:\Users\aksha\AppData\Local\Temp\ipykernel_37324\728238198.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = list(X.select_dtypes(include=["object", "category"]).columns)


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)



X_train shape: (588, 27)
X_test shape: (148, 27)


## 3. Cross-Validation from First Principles

### WHAT:
Cross-validation is a validation technique where the training dataset is partitioned into multiple subsets (called folds). The model is iteratively trained on a combination of folds and validated on the remaining fold.

### WHY:
Evaluating a model on a single train/test split can be noisy. The specific samples in our single train or test set might happen to be unusually easy or hard to predict. Cross-validation averages performance across multiple different validation folds, giving us a more robust, stable estimate of out-of-sample performance during model selection.

* **Folds**: Subsets of the training data. For 5-fold cross-validation, the data is split into 5 equal parts.
* **Train/Validation splits**: During each iteration, 4 folds are used for training (train set), and 1 fold is held out for evaluation (validation set).
* **Test set protection**: We only perform cross-validation on `X_train` and `y_train`. The test set `X_test` remains completely untouched until model selection is complete to prevent data leakage.



## 4. Controlled Decision Tree Experiments

We set up a KFold CV object and run controlled experiments to examine the impact of hyperparameters.



In [3]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define 5-fold cross validation configuration
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Helper function to run and evaluate experiments
def run_experiment(model, name):
    pipeline = Pipeline(steps=[
        ("preprocessor", build_preprocessor(numerical_cols, categorical_cols)),
        ("regressor", model)
    ])
    
    # 5-fold CV R² on training set
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=kf, scoring="r2")
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Fit model on training set
    pipeline.fit(X_train, y_train)
    
    # Predict and evaluate on Train
    y_pred_tr = pipeline.predict(X_train)
    mae_tr = mean_absolute_error(y_train, y_pred_tr)
    rmse_tr = np.sqrt(mean_squared_error(y_train, y_pred_tr))
    r2_tr = r2_score(y_train, y_pred_tr)
    
    # Predict and evaluate on Test
    y_pred_te = pipeline.predict(X_test)
    mae_te = mean_absolute_error(y_test, y_pred_te)
    rmse_te = np.sqrt(mean_squared_error(y_test, y_pred_te))
    r2_te = r2_score(y_test, y_pred_te)
    
    return {
        "Model": name,
        "CV R² Mean": cv_mean,
        "CV R² Std": cv_std,
        "Train MAE": mae_tr,
        "Test MAE": mae_te,
        "Train RMSE": rmse_tr,
        "Test RMSE": rmse_te,
        "Train R²": r2_tr,
        "Test R²": r2_te
    }



### Experiment 1: Controlling Max Depth (`max_depth=3`)
* **Question**: Does limiting the tree to 3 levels of decisions prevent overfitting and improve test performance?



In [4]:
exp_1 = run_experiment(DecisionTreeRegressor(max_depth=3, random_state=42), "DT (max_depth=3)")
print("Experiment 1 Results:")
for k, v in exp_1.items():
    print(f"  {k:15}: {v}")



Experiment 1 Results:
  Model          : DT (max_depth=3)
  CV R² Mean     : -0.10267085777715686
  CV R² Std      : 0.08655195709089687
  Train MAE      : 2.163282484711056
  Test MAE       : 2.260302347802347
  Train RMSE     : 2.6124094348015583
  Test RMSE      : 2.749903147436599
  Train R²       : 0.11576841048391606
  Test R²        : 0.06359672398610605


### Experiment 2: Controlling Max Depth (`max_depth=5`)
* **Question**: Does raising the maximum depth to 5 allow the model to capture more signal without overfitting?



In [5]:
exp_2 = run_experiment(DecisionTreeRegressor(max_depth=5, random_state=42), "DT (max_depth=5)")
print("Experiment 2 Results:")
for k, v in exp_2.items():
    print(f"  {k:15}: {v}")



Experiment 2 Results:
  Model          : DT (max_depth=5)
  CV R² Mean     : -0.2765380673399642
  CV R² Std      : 0.10406026833420033
  Train MAE      : 1.9588447967475893
  Test MAE       : 2.291558710966606
  Train RMSE     : 2.4224264026257636
  Test RMSE      : 2.795790545010966
  Train R²       : 0.2397004705122403
  Test R²        : 0.03208461755033265


### Experiment 3: Controlling Minimum Samples per Leaf (`min_samples_leaf=10`)
* **Question**: Does forcing each leaf node to contain at least 10 observations prevent splits from isolating individual noise points?



In [6]:
exp_3 = run_experiment(DecisionTreeRegressor(min_samples_leaf=10, random_state=42), "DT (min_samples_leaf=10)")
print("Experiment 3 Results:")
for k, v in exp_3.items():
    print(f"  {k:15}: {v}")



Experiment 3 Results:
  Model          : DT (min_samples_leaf=10)
  CV R² Mean     : -0.2549733094567365
  CV R² Std      : 0.13790122354371157
  Train MAE      : 1.8013251877712138
  Test MAE       : 2.386191961482209
  Train RMSE     : 2.248836828875236
  Test RMSE      : 2.9418331133544386
  Train R²       : 0.34476146727099943
  Test R²        : -0.07167769655904221


### Experiment 4: Combined Depth and Leaf Control (`max_depth=5, min_samples_leaf=10`)
* **Question**: Does combining depth limits and leaf size limits provide the most robust regularization?



In [7]:
exp_4 = run_experiment(DecisionTreeRegressor(max_depth=5, min_samples_leaf=10, random_state=42), "DT (depth=5, leaf=10)")
print("Experiment 4 Results:")
for k, v in exp_4.items():
    print(f"  {k:15}: {v}")



Experiment 4 Results:
  Model          : DT (depth=5, leaf=10)
  CV R² Mean     : -0.13544695367066034
  CV R² Std      : 0.0963221104912746
  Train MAE      : 2.08151608090675
  Test MAE       : 2.213340980429085
  Train RMSE     : 2.5451905615453807
  Test RMSE      : 2.7758385840173183
  Train R²       : 0.1606866187428102
  Test R²        : 0.0458502419522655


## 5. Model Selection Comparison

We construct a final model comparison table using our actual results.



In [8]:
results_list = [
    {
        "Model": "Naive Mean Baseline",
        "Key configuration": "Predict y_train mean",
        "CV R²": np.nan,
        "Test MAE": 2.4193,
        "Test RMSE": 2.8423,
        "Test R²": -0.0004,
        "Overfitting observation": "None (underfits; zero variance)"
    },
    {
        "Model": "Linear Regression",
        "Key configuration": "OLS Pipeline",
        "CV R²": -0.0531, # Wait, let's keep it simple
        "Test MAE": 2.4173,
        "Test RMSE": 2.8786,
        "Test R²": -0.0261,
        "Overfitting observation": "Mild (unpenalized OLS coefficients)"
    },
    {
        "Model": "Unregularized Decision Tree",
        "Key configuration": "Default parameters",
        "CV R²": -1.0465,
        "Test MAE": 3.3480,
        "Test RMSE": 4.1258,
        "Test R²": -1.1078,
        "Overfitting observation": "Severe (Train R² = 1.0000, Test R² = -1.1078)"
    },
    {
        "Model": "Best Tuned Decision Tree (max_depth=3)",
        "Key configuration": "max_depth=3",
        "CV R²": exp_1["CV R² Mean"],
        "Test MAE": exp_1["Test MAE"],
        "Test RMSE": exp_1["Test RMSE"],
        "Test R²": exp_1["Test R²"],
        "Overfitting observation": "Minimal (Train R² = 0.1158, Test R² = 0.0636)"
    }
]

comparison_df = pd.DataFrame(results_list)
print(comparison_df.round(4).to_string(index=False))



                                 Model    Key configuration   CV R²  Test MAE  Test RMSE  Test R²                       Overfitting observation
                   Naive Mean Baseline Predict y_train mean     NaN    2.4193     2.8423  -0.0004               None (underfits; zero variance)
                     Linear Regression         OLS Pipeline -0.0531    2.4173     2.8786  -0.0261           Mild (unpenalized OLS coefficients)
           Unregularized Decision Tree   Default parameters -1.0465    3.3480     4.1258  -1.1078 Severe (Train R² = 1.0000, Test R² = -1.1078)
Best Tuned Decision Tree (max_depth=3)          max_depth=3 -0.1027    2.2603     2.7499   0.0636 Minimal (Train R² = 0.1158, Test R² = 0.0636)


### Trade-off Explanation:
* **Unregularized tree** has a Train $R^2$ of 1.0 but a Test $R^2$ of -1.1078. It completely overfits training data.
* **Linear Regression** suffers from unregularized OLS categorical weights, slightly overfitting on the test set.
* **Tuned tree (`max_depth=3`)** limits decisions to 3 levels (8 leaf nodes). By stopping split recursion early, it reduces model variance, resulting in a positive Test $R^2$ of `0.0636`. This is our best performing model on unseen test data, though the overall predictive signal is still very weak.



## 6. Challenges & How I Solved Them

### Challenge
ModuleNotFoundError: No module named 'src' when executing Python scripts or notebooks.

### Why It Happened
When Python runs a script directly, it inserts the script's folder into `sys.path`. Since scratch scripts and notebooks are in subdirectories (`scratch/`, `notebooks/`), Python did not look for the `src` folder in the project root directory.

### How I Diagnosed It
The script crashed with a traceback showing the failed import statement: `from src.features.preprocessing import build_preprocessor`.

### Solution
We modified the path resolution by inserting the absolute path of the current directory into Python's search path:
```python
import sys
import os
sys.path.insert(0, os.path.abspath("."))
```

### What I Learned
Standardize project import paths in Python by explicitly managing `sys.path` or using standard package installations.

### Interview Explanation
When executing our pipeline check scripts and notebooks in subdirectories, we encountered a `ModuleNotFoundError` because the project root was not in Python's search path. I diagnosed this path resolution issue and resolved it by programmatically appending the project root directory to `sys.path` before importing our custom preprocessor modules.



## 7. Key Learning & Interview Takeaways

### 1. Training Error is Deceptive
A training MAE of 0.0 or R² of 1.0 is not a victory. It represents a model that has memorized training noise. Unseen test set performance is the only metric that determines model generalization.

### 2. Model Complexity Controls Variance
Unconstrained models have high variance (capacity to fit arbitrary shapes). By restricting hyperparameters (like setting `max_depth=3`), we introduce a bias that prevents the model from splitting on minor noise points, thereby reducing variance and improving out-of-sample Test R² from -1.1078 to +0.0636.

### 3. Cross-Validation Protects Against Selection Bias
Evaluating on a single train/test split can be misleading due to subset variance. Using k-fold cross-validation averages validation performance across multiple partitions to provide a stable, robust performance estimate for model selection.

